# Download WILD Aug16 Data → Process → Upload to HuggingFace

This notebook runs on **Kaggle** (free, fast internet) to:
1. Download 3 password-protected Aug16 feature files from UCSD SharePoint (~30GB each)
2. Process each into compressed `dataset_*.mat` files (~2-3GB each)
3. Upload processed files to HuggingFace `JustAGeek/dloc-wild-fig10b`
4. Delete raw file before downloading next (disk management)

After this, run evaluation/training on Vast.ai using the uploaded data.

In [ ]:
!pip install -q huggingface_hub h5py

import os, re, shutil, requests, time
import numpy as np
import h5py
from huggingface_hub import HfApi, hf_hub_download, login

# ============================================================
# CONFIG
# ============================================================
WILD_PASSWORD = "h!#ASe9018$%_rt#"
HF_DATASET_REPO = "JustAGeek/dloc-wild-fig10b"
# Paste your HuggingFace WRITE token here (Settings → Access Tokens → Write)
HF_TOKEN = "YOUR_HF_WRITE_TOKEN_HERE"

WORK_DIR = "/tmp/wild"
os.makedirs(WORK_DIR, exist_ok=True)
CHUNK = 300  # samples per read chunk

login(token=HF_TOKEN)
api = HfApi()
print(f"Logged in as: {api.whoami()['name']}")
print(f"Disk free: {shutil.disk_usage('/tmp').free / 1e9:.1f} GB")

In [ ]:
def download_wild_file(sharepoint_url, password, dest_path):
    """Download a password-protected file from UCSD SharePoint."""
    if os.path.exists(dest_path):
        print(f"  Already exists: {dest_path}")
        return True

    session = requests.Session()
    session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'})

    # Step 1: GET page, extract form fields
    print(f"  Authenticating...")
    r = session.get(sharepoint_url, allow_redirects=True)
    if r.status_code != 200:
        print(f"  ERROR: GET returned {r.status_code}")
        return False

    action = re.search(r'<form[^>]*action="([^"]+)"', r.text)
    if not action:
        print("  ERROR: No password form found")
        return False

    form_url = 'https://ucsdcloud-my.sharepoint.com' + action.group(1).replace('&amp;', '&')
    fields = {}
    for match in re.finditer(r'<input[^>]*name="([^"]+)"[^>]*value="([^"]*)"', r.text):
        fields[match.group(1)] = match.group(2)
    fields['txtPassword'] = password
    fields['btnSubmitPassword'] = 'Verify'

    # Step 2: POST with password
    print(f"  Submitting password...")
    r2 = session.post(form_url, data=fields, allow_redirects=False)
    if r2.status_code != 302 or 'Location' not in r2.headers:
        print(f"  ERROR: Auth failed (status {r2.status_code})")
        return False

    # Step 3: Follow redirect and stream download
    download_url = 'https://ucsdcloud-my.sharepoint.com' + r2.headers['Location']
    print(f"  Downloading...")
    r3 = session.get(download_url, stream=True, allow_redirects=True)
    if r3.status_code != 200:
        print(f"  ERROR: Download returned {r3.status_code}")
        return False

    total = int(r3.headers.get('Content-Length', 0))
    total_gb = total / 1e9
    print(f"  File size: {total_gb:.1f} GB")

    downloaded = 0
    t0 = time.time()
    with open(dest_path + '.partial', 'wb') as f:
        for chunk in r3.iter_content(chunk_size=64 * 1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            elapsed = time.time() - t0
            speed = downloaded / elapsed / 1e6
            pct = 100 * downloaded / total if total else 0
            print(f"  {downloaded/1e9:.1f}/{total_gb:.1f} GB ({pct:.0f}%) — {speed:.0f} MB/s", end='\r')
    print()

    os.rename(dest_path + '.partial', dest_path)
    print(f"  Saved: {dest_path} ({time.time()-t0:.0f}s)")
    return True


def process_features_to_dataset(raw_path, split_path, output_path):
    """Extract all samples from raw features file into compressed dataset .mat file.
    
    Uses split indices to combine fov_train + non_fov_train + fov_test + non_fov_test = ALL samples.
    Output format matches existing dataset_*.mat files (features_w_offset, features_wo_offset, labels_gaussian_2d).
    """
    print(f"  Loading split indices from {os.path.basename(split_path)}...")
    with h5py.File(split_path, 'r') as sf:
        all_idx = []
        for key in ['fov_train_idx', 'non_fov_train_idx', 'fov_test_idx', 'non_fov_test_idx']:
            idx = np.array(sf[key]).flatten().astype(np.int64) - 1  # MATLAB 1-based → 0-based
            all_idx.append(idx)
            print(f"    {key}: {len(idx)} samples")
        all_idx = np.sort(np.concatenate(all_idx))
    n = len(all_idx)
    print(f"  Total samples to extract: {n}")

    with h5py.File(raw_path, 'r') as src:
        keys = list(src.keys())
        w_key = 'features_w_offset' if 'features_w_offset' in keys else 'features_with_offset'
        wo_key = 'features_wo_offset' if 'features_wo_offset' in keys else 'features_without_offset'

        feat_shape = src[w_key].shape    # [361, 161, 4, N]
        lbl_shape = src['labels_gaussian_2d'].shape  # [361, 161, N]
        print(f"  Raw shapes: features={feat_shape}, labels={lbl_shape}")

        out_feat_shape = feat_shape[:-1] + (n,)
        out_lbl_shape = lbl_shape[:-1] + (n,)

        with h5py.File(output_path, 'w') as dst:
            ds_w = dst.create_dataset('features_w_offset', shape=out_feat_shape,
                                       dtype='float32', compression='gzip', chunks=True)
            ds_wo = dst.create_dataset('features_wo_offset', shape=out_feat_shape,
                                        dtype='float32', compression='gzip', chunks=True)
            ds_lbl = dst.create_dataset('labels_gaussian_2d', shape=out_lbl_shape,
                                         dtype='float32', compression='gzip', chunks=True)

            written = 0
            for start in range(0, n, CHUNK):
                end = min(start + CHUNK, n)
                chunk_idx = all_idx[start:end].tolist()
                bs = end - start

                ds_w[..., written:written+bs] = np.array(src[w_key][..., chunk_idx], dtype=np.float32)
                ds_wo[..., written:written+bs] = np.array(src[wo_key][..., chunk_idx], dtype=np.float32)
                ds_lbl[..., written:written+bs] = np.array(src['labels_gaussian_2d'][..., chunk_idx], dtype=np.float32)

                written += bs
                print(f"  Processed {written}/{n} samples", end='\r')
            print(f"  Processed {n}/{n} samples")

    size_gb = os.path.getsize(output_path) / 1e9
    print(f"  Output: {output_path} ({size_gb:.2f} GB)")
    return True

print("Functions defined.")

## Main Pipeline: Download → Process → Upload → Delete → Repeat

In [ ]:
# Download split index files from HuggingFace (uploaded earlier, ~30KB each)
print("Downloading split index files...")
split_files = {}
for name in ['jacobs_Aug16_1', 'jacobs_Aug16_3', 'jacobs_Aug16_4_ref']:
    split_files[name] = hf_hub_download(
        repo_id=HF_DATASET_REPO, repo_type="dataset",
        filename=f"data_split_idx/data_split_ids_{name}.mat")
    print(f"  {name}: OK")

# Sessions to process
SESSIONS = [
    ("jacobs_Aug16_1",
     "https://ucsdcloud-my.sharepoint.com/:u:/g/personal/sayyalas_ucsd_edu/EQ3xv70aECdDguiYTA3px0cBUQCJc5T7WFFrjeb67Ww2CA?download=1",
     "features_jacobs_Aug16_1.mat",
     "dataset_jacobs_Aug16_1.mat"),
    ("jacobs_Aug16_3",
     "https://ucsdcloud-my.sharepoint.com/:u:/g/personal/sayyalas_ucsd_edu/EUxwVT0kyFBKh2LY45vfrMYBlDApo4Alyr3xzyxOUsf0cQ?download=1",
     "features_jacobs_Aug16_3.mat",
     "dataset_jacobs_Aug16_3.mat"),
    ("jacobs_Aug16_4_ref",
     "https://ucsdcloud-my.sharepoint.com/:u:/g/personal/sayyalas_ucsd_edu/ETdyRRQe7UlJg5Aa5VsFf1ABsLj-aQWnRINB2VpHX_6XNw?download=1",
     "features_jacobs_Aug16_4_ref.mat",
     "dataset_jacobs_Aug16_4_ref.mat"),
]

results = {}

for session_name, url, raw_filename, out_filename in SESSIONS:
    print(f"\n{'='*70}")
    print(f"  Processing: {session_name}")
    print(f"{'='*70}")

    raw_path = os.path.join(WORK_DIR, raw_filename)
    out_path = os.path.join(WORK_DIR, out_filename)

    # 1. Download raw features from SharePoint
    print("\n[1/4] Downloading raw features...")
    ok = download_wild_file(url, WILD_PASSWORD, raw_path)
    if not ok:
        print(f"  FAILED — skipping {session_name}")
        results[session_name] = "DOWNLOAD FAILED"
        continue
    print(f"  Disk free: {shutil.disk_usage('/tmp').free / 1e9:.1f} GB")

    # 2. Process into compressed dataset
    print(f"\n[2/4] Processing into {out_filename}...")
    process_features_to_dataset(raw_path, split_files[session_name], out_path)

    # 3. Delete raw file to free disk
    print(f"\n[3/4] Deleting raw file ({os.path.getsize(raw_path)/1e9:.1f} GB)...")
    os.remove(raw_path)
    print(f"  Disk free: {shutil.disk_usage('/tmp').free / 1e9:.1f} GB")

    # 4. Upload processed file to HuggingFace
    out_size = os.path.getsize(out_path) / 1e9
    print(f"\n[4/4] Uploading {out_filename} ({out_size:.2f} GB) to HuggingFace...")
    api.upload_file(
        path_or_fileobj=out_path,
        path_in_repo=f"data/{out_filename}",
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
    )
    print(f"  Uploaded!")

    # Delete processed file too (free disk for next session)
    os.remove(out_path)
    results[session_name] = f"OK ({out_size:.2f} GB)"
    print(f"  Disk free: {shutil.disk_usage('/tmp').free / 1e9:.1f} GB")

# Summary
print(f"\n{'='*70}")
print("  DONE — Summary")
print(f"{'='*70}")
for name, status in results.items():
    print(f"  {name}: {status}")
print(f"\nFiles uploaded to: https://huggingface.co/datasets/{HF_DATASET_REPO}")
print("Now download them on Vast.ai with:")
print(f"  huggingface-cli download {HF_DATASET_REPO} --local-dir ./data --repo-type dataset")